In [0]:
class invoiceStream():
    def __init__(self):
        self.base_data_dir = "/Volumes/streamingdata/dbo/"

    def getSchema(self):
        return """
            InvoiceNumber string, CreatedTime bigint, StoreID string, PosID string, CashierID string, CustomerType string, CustomerCardno string, TotalAmount double, NumberOfItems bigint, PaymentMethod string, TaxableAmount double, CGST double, SGST double, CESS double, DeliveryType string,
            DeliveryAddress struct<AddressLine string, City string, ContactNumber string, PinCode string, State string>,
            InvoiceLineItems array<struct<ItemCode string, ItemDescription string, ItemPrice double, ItemQty bigint, TotalValue double>>
            """
    
    def readInvoices(self):
        return (
            spark.readStream.format("json")
                    .schema(self.getSchema())
                    .load(f"{self.base_data_dir}/data/invoices")
        )

    def explodeInvoices(self, invoiceDF):
        return (
            invoiceDF.selectExpr(
                "InvoiceNumber",
                "CreatedTime",
                "StoreID",
                "PosID",
                "CustomerType",
                "PaymentMethod",
                "DeliveryType",
                "DeliveryAddress.City",
                "DeliveryAddress.State",
                "DeliveryAddress.PinCode",
                "explode(InvoiceLineItems) as LineItem"
                )
        )

    def flattenInvoices(self, explodeDF):
        from pysopark.sql.functions import expr
        return (
            explodeDF.withColumn("ItemCode", expr("LineItem.ItemCode"))
                    .withColumn("ItemDescription", expr("LineItem.ItemDescription"))
                    .withColumn("ItemPrice", expr("LineItem.ItemPrice"))
                    .withColumn("ItemQty", expr("LineItem.ItemQty"))
                    .withColumn("TotalValue", expr("LineItem.TotalValue"))
                    .drop("LineItem")
        )
    
    def appendInvoices(self, flattenedDF):
        return (
            flattenedDF.writeStream
                    .format("delta")
                    .option("checkpointLocation", f"{self.base_data_dir}/checkpoint/invoices")
                    .trigger(once=True)
                    .outputMode("append")
                    .toTable("streamingdata.dbo.invoice_line_item")
        )

    def process(self):
        print(f"Staring Invoice Processing Stream...", end='')
        invoicesDF = self.readInvoices()
        explodeDF = self.explodeInvoices(invoicesDF)
        resultDF = self.flattenInvoices(explodeDF)
        sQuery = self.appendInvoices(resultDF)
        print("Done/n")
        return sQuery

    

In [0]:
rom py